# Business Entity Resolution: v8 (richer features + a team of 4 boosting models)

**How to run:** click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**.
Settings: CPU (no accelerator), Internet On. It runs on Kaggle's servers even if you close the tab.

What is new compared with v1 (same shortlist as v1):
1. **Sound-alike codes:** each name word becomes a "sounds like" code (metaphone), so *Sharma* and *Sharmaa* match.
2. **Rare words count more:** words shared by thousands of businesses ("store", "road") count little; rare words count a lot.
   Also: "the rarest word of the business that the candidate is missing" (a strong warning sign).
3. **Rank features:** how each candidate compares with the same business's other candidates (rank, gap to the best, lead over the runner-up).
4. **4 models instead of 1:** two LightGBM (different settings), XGBoost and CatBoost. Each gives a probability;
   the notebook tries their average, the average of the best 3, and the best single model, and keeps whichever scores best on held-out businesses.

Cells: 1 install, 2 v1 pipeline code, 3 v8 code, 4 settings, 5 **train** (prints `BEST val F0.5` for every model and blend; v1 was 0.9529),
6 **predict** test and write `output/matching_results.tsv`, 7 official format check.

No external data or APIs are used, only the competition files. Licenses: LightGBM MIT, XGBoost Apache-2.0, CatBoost Apache-2.0, jellyfish MIT.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm xgboost catboost jellyfish

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr)))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3)."""
    def __init__(self, B, k_state=15, k_unk=10):
        self.B, self.k_state, self.k_unk = B, k_state, k_unk
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in (("name_skc", _vec_name), ("addr_sk", _vec_addr)):
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos])
                entry[col] = (v, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr); a = row in A, b = row in self.B."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for col, tag in (("name_skc", 0), ("addr_sk", 1)):
                if e[col] is None:
                    continue
                v, BT = e[col]
                texts = A[col].to_numpy()[rows]
                for i in range(0, len(rows), chunk):
                    C = sp_matmul_topn(v.transform(texts[i:i + chunk]), BT, top_n=k, threshold=thr,
                                       n_threads=N_CPU).tocoo()
                    out.append((rows[i + C.row], e["pos"][C.col], C.data, tag))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": []})
        a = np.concatenate([o[0] for o in out]).astype(np.int64)
        b = np.concatenate([o[1] for o in out]).astype(np.int64)
        sim = np.concatenate([o[2] for o in out]).astype(np.float32)
        tag = np.concatenate([np.full(len(o[0]), o[3], np.int8) for o in out])
        df = pd.DataFrame({"key": a * len(self.B) + b,
                           "sim_name": np.where(tag == 0, sim, np.nan),
                           "sim_addr": np.where(tag == 1, sim, np.nan)})
        df = df.groupby("key", sort=False).max().reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")


In [ ]:
# =====================================================================
# v8 additions: richer pair features + an ensemble of gradient boosting models.
# Everything here works on top of the v1 pipeline (erpipe) and uses no external data.
# Libraries: jellyfish (MIT), xgboost (Apache-2.0), catboost (Apache-2.0), lightgbm (MIT).
# =====================================================================
import jellyfish
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
import xgboost as xgb
from catboost import CatBoostClassifier

# ---------------------------------------------------------------------
# 1. Extra cleaned columns per record
#    name_w  : the name's words without legal words (Pvt, Ltd, LLC, SARL ...)
#    name_ph : the same words as sound codes (metaphone), so "Sharma" and "Sharmaa" give the same code
# ---------------------------------------------------------------------
def _extra_rows(names):
    out = []
    for nb in names:
        toks = nb.split()
        core = [t for t in toks if skel_tok(t) not in LEGAL_SK] or toks
        ph = [jellyfish.metaphone(t) if not t.isdigit() else t for t in core]
        out.append((" ".join(core), " ".join(p for p in ph if p)))
    return out

def prepare_extra(df):
    names = df.name_b.tolist(); step = 100_000
    with Pool(N_CPU) as pool:
        res = pool.map(_extra_rows, [names[i:i + step] for i in range(0, len(names), step)])
    res = [r for chunk in res for r in chunk]
    df["name_w"] = [r[0] for r in res]
    df["name_ph"] = [r[1] for r in res]
    return df

# ---------------------------------------------------------------------
# 2. Word-rarity ("IDF") weights, learned from the candidate pool itself.
#    A word shared by 50,000 businesses ("store", "road") says little; a rare word ("zanskar") says a lot.
# ---------------------------------------------------------------------
class Rarity:
    """Word vocabularies + rarity weights for one candidate pool (one source, one country)."""
    def __init__(self, B):
        self.parts = {}
        for col in ("name_w", "addr_n", "name_ph"):
            v = CountVectorizer(token_pattern=r"\S+", binary=True, dtype=np.float32)
            try:
                X = v.fit_transform(B[col].to_numpy())
            except ValueError:            # empty vocabulary
                self.parts[col] = None; continue
            df = np.asarray(X.sum(axis=0)).ravel()
            idf = (np.log((len(B) + 1) / (df + 1)) + 1).astype(np.float32)
            self.parts[col] = (v, idf, X.tocsr(), float(np.log(len(B) + 1) + 1))
        # full char 3-gram similarity of the name skeleton for EVERY pair (the shortlist only kept top-15)
        self.cv = TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
        try:
            self.CB = self.cv.fit_transform(B.name_skc.to_numpy()).tocsr()
        except ValueError:
            self.cv = None

def _rowdot(XA, XB):
    return np.asarray(XA.multiply(XB).sum(axis=1)).ravel().astype(np.float32)

def _rowmax(X):
    if X.nnz == 0:
        return np.zeros(X.shape[0], np.float32)
    return np.asarray(X.max(axis=1).todense()).ravel().astype(np.float32)

def rarity_features(R, a, b, A, f, chunk=3_000_000):
    """Adds rarity-weighted similarities for (A row a, pool row b) pairs into DataFrame f."""
    for col, pre in (("name_w", "rn"), ("addr_n", "ra"), ("name_ph", "rp")):
        names = [f"{pre}_{x}" for x in ("wjacc", "cont_s1", "cont_c", "cos", "miss_s1", "miss_c", "share_max")]
        if R.parts[col] is None:
            for n in names: f[n] = np.float32(np.nan)
            continue
        v, idf, XB_all, max_idf = R.parts[col]
        texts = A[col].to_numpy()
        XA_all = v.transform(texts).tocsr()
        # words of the S1 record that never occur in the pool: maximally rare, and surely not shared
        n_unseen = np.array([len(set(t.split())) for t in texts], np.float32) - np.asarray(XA_all.sum(axis=1)).ravel()
        W = sp.diags(idf); W2 = sp.diags(idf * idf)
        XAw, XBw = (XA_all @ W).tocsr(), (XB_all @ W).tocsr()
        sA = np.asarray(XAw.sum(axis=1)).ravel() + n_unseen * max_idf
        sB = np.asarray(XBw.sum(axis=1)).ravel()
        nA = np.sqrt(np.asarray((XA_all @ W2).sum(axis=1)).ravel() + n_unseen * max_idf ** 2)
        nB = np.sqrt(np.asarray((XB_all @ W2).sum(axis=1)).ravel())
        res = {n: np.empty(len(a), np.float32) for n in names}
        for i in range(0, len(a), chunk):
            ai, bi = a[i:i + chunk], b[i:i + chunk]
            PA, PB = XAw[ai], XB_all[bi]
            S = PA.multiply(PB).tocsr()                               # shared words, with their weights
            inter = np.asarray(S.sum(axis=1)).ravel()
            inter2 = np.asarray(S.multiply(S).sum(axis=1)).ravel()
            sa, sb = sA[ai], sB[bi]
            with np.errstate(divide="ignore", invalid="ignore"):
                res[names[0]][i:i + chunk] = np.where(sa + sb - inter > 0, inter / (sa + sb - inter), np.nan)
                res[names[1]][i:i + chunk] = np.where(sa > 0, inter / sa, np.nan)
                res[names[2]][i:i + chunk] = np.where(sb > 0, inter / sb, np.nan)
                res[names[3]][i:i + chunk] = np.where(nA[ai] * nB[bi] > 0, inter2 / (nA[ai] * nB[bi]), np.nan)
            missA = _rowmax((PA - PA.multiply(PB > 0)).tocsr())        # rarest S1 word the candidate lacks
            res[names[4]][i:i + chunk] = np.maximum(missA, np.where(n_unseen[ai] > 0, max_idf, 0))
            PBw = XBw[bi]
            res[names[5]][i:i + chunk] = _rowmax((PBw - PBw.multiply(PA > 0)).tocsr())   # rarest candidate word S1 lacks
            res[names[6]][i:i + chunk] = _rowmax(S)                   # rarest word they share
        for n in names: f[n] = res[n]
    if R.cv is not None:
        CA = R.cv.transform(A.name_skc.to_numpy()).tocsr()
        out = np.empty(len(a), np.float32)
        for i in range(0, len(a), chunk):
            out[i:i + chunk] = _rowdot(CA[a[i:i + chunk]], R.CB[b[i:i + chunk]])
        f["nch_cos"] = out
    else:
        f["nch_cos"] = np.float32(np.nan)

def phonetic_features(a, b, A, B, f):
    x, y = A.name_ph.to_numpy()[a].tolist(), B.name_ph.to_numpy()[b].tolist()
    f["ph_tset"] = cpdist(x, y, scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32)
    f["ph_ratio"] = cpdist(x, y, scorer=fuzz.ratio, workers=N_CPU).astype(np.float32)
    f["ph_first_eq"] = np.array([(p.split(" ", 1)[0] == q.split(" ", 1)[0]) if p and q else -1
                                 for p, q in zip(x, y)], np.int8)
    xw, yw = A.name_w.to_numpy()[a].tolist(), B.name_w.to_numpy()[b].tolist()
    f["nw_tset"] = cpdist(xw, yw, scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32)

# ---------------------------------------------------------------------
# 3. Rank features: how does this candidate compare with the business's other candidates (same source)?
# ---------------------------------------------------------------------
RANK_ON = ["rn_cos", "ra_cos", "nch_cos", "ph_tset", "combo"]

def rank_features(a, f):
    f["combo"] = (f.rn_cos.fillna(0) + f.ra_cos.fillna(0) + f.nch_cos.fillna(0) + f.ph_tset / 100).astype(np.float32)
    g = pd.DataFrame({"a": a})
    for k in RANK_ON:
        g["v"] = f[k].fillna(-1).to_numpy()
        grp = g.groupby("a")["v"]
        mx = grp.transform("max").to_numpy()
        f[f"rk_{k}"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
        f[f"gap_{k}"] = (mx - g.v.to_numpy()).astype(np.float32)
    v = g.v.to_numpy()                      # g.v is "combo" now (last in RANK_ON)
    # runner-up value per business, found by sorting once
    order = np.lexsort((-v, a)); a_s, v_s = a[order], v[order]
    first = np.r_[True, a_s[1:] != a_s[:-1]]
    pos = np.arange(len(a_s)); start = np.maximum.accumulate(np.where(first, pos, 0))
    has2 = (start + 1 < len(a_s)) & (a_s[np.minimum(start + 1, len(a_s) - 1)] == a_s)
    sec_s = np.where(has2, v_s[np.minimum(start + 1, len(a_s) - 1)], -1)
    second = np.empty_like(v); second[order] = sec_s
    # top candidate: lead over the runner-up; others: distance behind the leader (negative)
    f["combo_lead"] = np.where(v >= mx, v - second, v - mx).astype(np.float32)
    f["combo_z"] = ((g.v - grp.transform("mean")) / (grp.transform("std").fillna(0) + 1e-3)).to_numpy(np.float32)
    f["n_close"] = g.assign(c=(mx - v) < 0.15).groupby("a")["c"].transform("sum").to_numpy(np.int16)

def features_v8(P, A, B, is_s3, R):
    """v1 features + v8 features for the shortlist pairs P of A (S1 batch) against pool B."""
    f = features(P, A, B, is_s3)
    a, b = P.a.to_numpy(), P.b.to_numpy()
    rarity_features(R, a, b, A, f)
    phonetic_features(a, b, A, B, f)
    rank_features(a, f)
    return f

# ---------------------------------------------------------------------
# 4. The ensemble: several boosting models, each gives a probability; the blend is their average.
# ---------------------------------------------------------------------
MODEL_SPECS = [
    ("lgb_a", "lgb", dict(learning_rate=0.05, num_leaves=255, min_child_samples=100, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=0)),
    ("lgb_b", "lgb", dict(learning_rate=0.05, num_leaves=127, min_child_samples=50, subsample=0.7, subsample_freq=1,
                          colsample_bytree=0.5, reg_lambda=5.0, random_state=1)),
    ("xgb", "xgb", dict(eta=0.08, max_depth=9, min_child_weight=5, subsample=0.8, colsample_bytree=0.7,
                        tree_method="hist", max_bin=256, eval_metric="logloss", objective="binary:logistic", seed=2)),
    ("cat", "cat", dict(learning_rate=0.12, depth=8, l2_leaf_reg=3, random_seed=3, border_count=128)),
]

def fit_model(kind, params, Xtr, ytr, Xva, yva, max_rounds, n_rounds=None):
    """Trains one model. With n_rounds=None it stops when the held-out loss stops improving."""
    es = n_rounds is None
    rounds = max_rounds if es else n_rounds
    if kind == "lgb":
        m = lgb.LGBMClassifier(n_estimators=rounds, n_jobs=N_CPU, verbose=-1, **params)
        m.fit(Xtr, ytr, **({"eval_set": [(Xva, yva)], "callbacks": [lgb.early_stopping(100, verbose=False)]} if es else {}))
        return m, (m.best_iteration_ if es else rounds)
    if kind == "xgb":
        dtr = xgb.QuantileDMatrix(Xtr, ytr, max_bin=params.get("max_bin", 256), nthread=N_CPU)
        ev = [(xgb.DMatrix(Xva, yva, nthread=N_CPU), "val")] if es else []
        m = xgb.train({**params, "nthread": N_CPU}, dtr, rounds, evals=ev,
                      early_stopping_rounds=100 if es else None, verbose_eval=False)
        del dtr; gc.collect()
        return m, (m.best_iteration + 1 if es else rounds)
    if kind == "cat":
        m = CatBoostClassifier(iterations=rounds, thread_count=N_CPU, verbose=False, allow_writing_files=False,
                               **({"od_type": "Iter", "od_wait": 100, "use_best_model": True} if es else {}), **params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva) if es else None)
        return m, (m.get_best_iteration() + 1 if es else rounds)

def predict_model(kind, m, X):
    if kind == "lgb":
        return m.predict(X, num_threads=N_CPU).astype(np.float32) if isinstance(m, lgb.Booster) \
            else m.predict_proba(X)[:, 1].astype(np.float32)
    if kind == "xgb":
        return m.predict(xgb.DMatrix(X, nthread=N_CPU)).astype(np.float32)
    return m.predict_proba(X)[:, 1].astype(np.float32)

def save_model(kind, m, path):
    if kind == "lgb": m.booster_.save_model(path)
    elif kind == "xgb": m.save_model(path)
    else: m.save_model(path)

def load_model(kind, path):
    if kind == "lgb": return lgb.Booster(model_file=path)
    if kind == "xgb":
        m = xgb.Booster(); m.load_model(path); m.set_param({"nthread": N_CPU}); return m
    m = CatBoostClassifier(); m.load_model(path); return m

MODEL_EXT = {"lgb": "txt", "xgb": "json", "cat": "cbm"}


In [ ]:
ROOT = os.environ.get("ROOT", "/kaggle/input")      # the notebook searches here for the .tsv files
WORK = os.environ.get("WORK", "/kaggle/working")    # models + outputs are saved here
N_TRAIN_S1 = int(os.environ.get("N_TRAIN_S1", 150_000))  # training businesses (same 150k as v1, so scores compare)
BATCH = int(os.environ.get("BATCH", 100_000))       # test businesses processed at once (lower it if memory runs out)
MAX_ROUNDS = int(os.environ.get("MAX_ROUNDS", 3000))    # upper limit of trees per model (early stopping picks the number)
REFIT = int(os.environ.get("REFIT", 1))             # 1 = retrain each model on all 150k after choosing its tree count
SMOKE = 0             # 1 = tiny quick run on a slice of the data (checks the code only)
os.makedirs(WORK, exist_ok=True)
log("CPU cores:", N_CPU, "| training businesses:", N_TRAIN_S1, "| refit:", bool(REFIT), "| smoke test:", bool(SMOKE))


In [ ]:
# ---------------------------------------------------------------------
# TRAIN: build pairs + features for 150k training businesses, train 4 boosting models,
# measure each one and their blend on 20% held-out businesses (the same ones v1 used).
# ---------------------------------------------------------------------
T0 = time.time()
if not os.path.exists(f"{WORK}/v8_cache.json"):      # features are cached on disk; a re-run skips this part
    log("loading train Source 1 + ground truth")
    s1 = load(ROOT, "train_source1"); gt = load(ROOT, "train_ground_truth")
    s1 = s1.sample(4000 if SMOKE else N_TRAIN_S1, random_state=0).reset_index(drop=True)
    truth = {r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x) for r in gt.itertuples()}
    del gt; gc.collect()
    A = prepare_extra(prepare(s1)); log("prepared S1 sample", A.shape)
    feats, metas = [], []
    for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
        log("loading", src); B = load(ROOT, src)
        if SMOKE:   # keep the true matches of the sample + 200k random records, so the shortlist still has look-alikes
            want = set().union(*(truth[s] for s in A.entity_id))
            B = pd.concat([B[B.entity_id.isin(want)], B.sample(200_000, random_state=0)]).drop_duplicates("entity_id")
            B = B.reset_index(drop=True)
        B = prepare_extra(prepare(B)); log("prepared", src, B.shape)
        for country in A.country.unique():
            Ac = A[A.country == country].reset_index(drop=True); Bc = B[B.country == country].reset_index(drop=True)
            if len(Ac) == 0 or len(Bc) == 0: continue
            P = Index(Bc).query(Ac); R = Rarity(Bc); log(src, country, "pairs", len(P))
            F = features_v8(P, Ac, Bc, is_s3, R); del R
            m = pd.DataFrame({"s1_id": Ac.entity_id.to_numpy()[P.a], "cand_id": Bc.entity_id.to_numpy()[P.b]})
            m["label"] = [c in truth.get(s, ()) for s, c in zip(m.s1_id, m.cand_id)]
            fp = f"{WORK}/v8_feat_{len(feats)}.parquet"; F.to_parquet(fp)   # parked on disk to keep memory free
            feats.append((fp, len(F))); metas.append(m); log(src, country, "features done", F.shape)
            del Ac, Bc, P, F, m; gc.collect()          # Bc still points at B's text; free it before the next source
        del B; gc.collect()
    M = pd.concat(metas, ignore_index=True); del metas; gc.collect()
    FEATS = list(pd.read_parquet(feats[0][0]).columns)
    y_all = M.label.to_numpy().astype(np.int8)
    all_true = sum(len(truth[s]) for s in A.entity_id)
    log("pairs", len(M), "per S1", round(len(M) / len(A), 1), "| shortlist recall", round(y_all.sum() / all_true, 4),
        "| features", len(FEATS))

    # held-out businesses: same random choice as v1 (seed 42, 20%)
    rng = np.random.default_rng(42)
    val_ids = set(rng.choice(A.entity_id.to_numpy(), int(0.2 * len(A)), replace=False))
    iv = M.s1_id.isin(val_ids).to_numpy()
    perm = np.r_[np.flatnonzero(~iv), np.flatnonzero(iv)]; n_tr = int((~iv).sum())
    dest = np.empty(len(perm), np.int64); dest[perm] = np.arange(len(perm))
    Xn = np.lib.format.open_memmap(f"{WORK}/v8_X.npy", "w+", np.float32, (len(perm), len(FEATS)))  # on disk, training rows first
    o = 0
    while feats:                                                   # move piece by piece, so memory is not doubled
        fp, n = feats.pop(0); F = pd.read_parquet(fp)
        Xn[dest[o:o + n]] = F[FEATS].to_numpy(np.float32); o += n; del F; os.remove(fp); gc.collect()
    del dest; Xn.flush(); del Xn; gc.collect()
    np.save(f"{WORK}/v8_y.npy", y_all[perm])
    M.iloc[perm[n_tr:]][["s1_id", "cand_id"]].reset_index(drop=True).to_parquet(f"{WORK}/v8_Mv.parquet")
    json.dump({"FEATS": FEATS, "n_tr": n_tr, "vt": {s: sorted(truth[s]) for s in val_ids}}, open(f"{WORK}/v8_cache.json", "w"))
    del M, truth, A; gc.collect()

C = json.load(open(f"{WORK}/v8_cache.json")); FEATS, n_tr = C["FEATS"], C["n_tr"]
vt = {s: set(v) for s, v in C["vt"].items()}; vl = list(vt); del C
Xn = np.load(f"{WORK}/v8_X.npy", mmap_mode="r")        # read from disk on demand, so it does not fill the memory
y = np.load(f"{WORK}/v8_y.npy"); Mv = pd.read_parquet(f"{WORK}/v8_Mv.parquet")
log("features ready:", Xn.shape, "| minutes so far", round((time.time() - T0) / 60, 1))
Xtr, ytr, Xva, yva = Xn[:n_tr], y[:n_tr], Xn[n_tr:], y[n_tr:]
log("baseline all-empty", round(macro_f05({}, vt, vl), 4))
log("ceiling (perfect model on this shortlist)",
    round(macro_f05(Mv[yva == 1].groupby("s1_id").cand_id.apply(set).to_dict(), vt, vl), 4))
GRID = np.round(np.arange(0.30, 0.901, 0.025), 3)

def best_cut(p, label):
    pv = Mv.assign(p=p)
    res = [(t, macro_f05(decide(pv, t), vt, vl)) for t in GRID]
    t, s = max(res, key=lambda r: r[1])
    log(f"BEST val F0.5 [{label}] = {s:.4f} at cut-off {t}")
    return float(t), float(s)

PV, RESULTS, ROUNDS = {}, {}, {}
for name, kind, params in MODEL_SPECS:
    t1 = time.time(); log("training", name)
    m, n = fit_model(kind, params, Xtr, ytr, Xva, yva, MAX_ROUNDS)
    PV[name] = predict_model(kind, m, Xva); ROUNDS[name] = n
    log(name, "trees", n, "| minutes", round((time.time() - t1) / 60, 1))
    RESULTS[name] = best_cut(PV[name], name)
    if not REFIT:
        save_model(kind, m, f"{WORK}/v8_{name}.{MODEL_EXT[kind]}")
    del m; gc.collect()

# blends: the average of all models, and the average of the 3 best (in case one model drags the others down)
order = sorted(PV, key=lambda k: -RESULTS[k][1])
cands = {"blend_all": list(PV), "blend_top3": order[:3], "best_single": order[:1]}
for lab, members in cands.items():
    RESULTS[lab] = best_cut(np.mean([PV[k] for k in members], axis=0), f"{lab}: {'+'.join(members)}")
choice = max(cands, key=lambda k: RESULTS[k][1])
USE, CUT = cands[choice], RESULTS[choice][0]
log(f"FINAL choice: {choice} = {'+'.join(USE)}, cut-off {CUT}, val F0.5 {RESULTS[choice][1]:.4f} (v1 was 0.9529)")
del PV; gc.collect()

if REFIT:   # retrain the chosen models on all 150k businesses with the tree counts found above
    for name, kind, params in MODEL_SPECS:
        if name not in USE: continue
        t1 = time.time(); log("refit on all training businesses:", name, ROUNDS[name], "trees")
        m, _ = fit_model(kind, params, Xn, y, None, None, MAX_ROUNDS, n_rounds=ROUNDS[name])
        save_model(kind, m, f"{WORK}/v8_{name}.{MODEL_EXT[kind]}"); del m; gc.collect()
        log("refit done", name, "| minutes", round((time.time() - t1) / 60, 1))
json.dump({"features": FEATS, "models": [(n, k) for n, k, _ in MODEL_SPECS if n in USE], "threshold": CUT,
           "choice": choice, "results": RESULTS, "rounds": ROUNDS},
          open(f"{WORK}/v8_meta.json", "w"), indent=1)
del Xn, Xtr, Xva, y; gc.collect()
log("training finished | total minutes", round((time.time() - T0) / 60, 1))


In [ ]:
# ---------------------------------------------------------------------
# PREDICT: every test business, in batches saved to disk (safe to resume), then write both TSVs.
# ---------------------------------------------------------------------
T0 = time.time()
meta = json.load(open(f"{WORK}/v8_meta.json"))
MODELS = [(n, k, load_model(k, f"{WORK}/v8_{n}.{MODEL_EXT[k]}")) for n, k in meta["models"]]
log("models:", [n for n, _, _ in MODELS], "| cut-off", meta["threshold"])
PARTS = f"{WORK}/parts_v8"; os.makedirs(PARTS, exist_ok=True)
log("loading test Source 1"); A_full = load(ROOT, "test_source1")
s1_ids = A_full.entity_id.to_numpy()
A = A_full.groupby("country").head(1500).reset_index(drop=True) if SMOKE else A_full
del A_full; gc.collect()
A = prepare_extra(prepare(A)); log("prepared", A.shape)
print(A.country.value_counts().to_string())
row_of = {s: i for i, s in enumerate(s1_ids)}
A_rowid = np.array([row_of[s] for s in A.entity_id], np.int32)
for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    todo = [(c, i) for c in A.country.unique() for i in range(0, int((A.country == c).sum()), BATCH)
            if not os.path.exists(f"{PARTS}/{src}_{c}_{i}.parquet")]
    if not todo:
        log(src, "already done"); continue
    log("loading", src); B = load(ROOT, src)
    if SMOKE: B = B.sample(300_000, random_state=0).reset_index(drop=True)
    B = prepare_extra(prepare(B)); log("prepared", src, B.shape)
    for country in A.country.unique():                 # any label works, France included
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        Bc = B[B.country == country].reset_index(drop=True)
        if len(Bc) == 0 or not any(c == country for c, _ in todo): continue
        idx = Index(Bc); R = Rarity(Bc); bids = Bc.entity_id.to_numpy()
        nb = -(-len(a_rows) // BATCH)
        for i in range(0, len(a_rows), BATCH):
            out = f"{PARTS}/{src}_{country}_{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P = idx.query(Ac)
            if len(P):
                F = features_v8(P, Ac, Bc, is_s3, R)[meta["features"]].to_numpy(np.float32)
                p = np.mean([predict_model(k, m, F) for _, k, m in MODELS], axis=0).astype(np.float32)
                del F
            else:
                p = np.zeros(0, np.float32)
            pd.DataFrame({"s1_row": A_rowid[rows[P.a.to_numpy()]], "cand_id": bids[P.b.to_numpy()], "p": p}) \
                .to_parquet(out)
            log(src, country, f"batch {i // BATCH + 1}/{nb}", "pairs", len(P),
                "| elapsed min", round((time.time() - T0) / 60, 1))
            del P, p; gc.collect()
        del idx, R, Bc; gc.collect()
    del B; gc.collect()

log("combining batches (low-memory)")
del A; gc.collect()
# 1) final matches: only pairs above the cut-off are loaded; each S2/S3 record goes to its best S1 only
best = (pl.scan_parquet(f"{PARTS}/*.parquet").filter(pl.col("p") >= meta["threshold"]).collect()
        .sort("p", descending=True).unique(subset="cand_id", keep="first", maintain_order=True))
match = best.group_by("s1_row").agg(pl.col("cand_id").sort())
mline = np.full(len(s1_ids), "", dtype=object)
for r, c in zip(match["s1_row"].to_list(), match["cand_id"].to_list()):
    mline[r] = ",".join(c)
n = int((mline != "").sum()); n_links = best.height; del best, match; gc.collect()
os.makedirs(f"{WORK}/output", exist_ok=True)
with open(f"{WORK}/output/matching_results.tsv", "w", encoding="utf-8") as fh:
    fh.write("source1_entity_id\tmatched_entity_ids\n")
    for s, m in zip(s1_ids, mline):
        fh.write(s + "\t" + m + "\n")
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches",
    round(n_links / len(s1_ids), 2), "| predict minutes", round((time.time() - T0) / 60, 1))
del mline; gc.collect()
# 2) candidate list (the shortlist), one batch at a time
cline = np.full(len(s1_ids), "", dtype=object)
for f in sorted(glob.glob(f"{PARTS}/*.parquet")):
    g = pl.read_parquet(f, columns=["s1_row", "cand_id"]).group_by("s1_row").agg(pl.col("cand_id").unique(maintain_order=True))
    for r, c in zip(g["s1_row"].to_list(), g["cand_id"].to_list()):
        cline[r] = (cline[r] + "," if cline[r] else "") + ",".join(c)
    del g; gc.collect()
with open(f"{WORK}/output/candidate_pairs.tsv", "w", encoding="utf-8") as fh:
    fh.write("source1_entity_id\tcandidate_entity_ids\n")
    for s, c in zip(s1_ids, cline):
        fh.write(s + "\t" + c + "\n")
del cline; gc.collect()
log("candidate file written")


In [ ]:
# the candidate file is skipped here: checking it needs a lot of memory (the file is still written)
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
!python3 {validator} --matching {WORK}/output/matching_results.tsv --test-dir {test_dir} --candidate none.tsv

**Done.** Open this version's **Output** tab, go into `output/`, and download **`matching_results.tsv`**. Keep `candidate_pairs.tsv` for the final zip. You can delete the `parts_v8/` folder.